In [24]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

Шаг 1. Загрузка данных

In [2]:
df = pd.read_csv('data/song_dataset.csv')
df.head()

,song_popularity,song_duration_ms,acousticness,danceability,energy,instrumentalness,key,liveness,loudness,audio_mode,speechiness,tempo,time_signature,audio_valence
0,73,262333,0.005520,0.496,0.682,0.000029,8,0.0589,-4.095,1,0.0294,167.060,4,0.474
1,66,216933,0.010300,0.542,0.853,0.000000,3,0.1080,-6.407,0,0.0498,105.256,4,0.370
2,76,231733,0.008170,0.737,0.463,0.447000,0,0.2550,-7.828,1,0.0792,123.881,4,0.324
3,74,216933,0.026400,0.451,0.970,0.003550,0,0.1020,-4.938,1,0.1070,122.444,4,0.198
4,56,223826,0.000954,0.447,0.766,0.000000,10,0.1130,-5.065,1,0.0313,172.011,4,0.574


In [3]:
# Посмотрим на форму датасета
print(f"\nРазмер выборки: {df.shape}\n")

# Выведем основную информацию о колонках
print(df.info())


Размер выборки: (1899, 14)

<class 'pandas.DataFrame'>
RangeIndex: 1899 entries, 0 to 1898
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   song_popularity   1899 non-null   int64  
 1   song_duration_ms  1899 non-null   int64  
 2   acousticness      1899 non-null   float64
 3   danceability      1899 non-null   float64
 4   energy            1899 non-null   float64
 5   instrumentalness  1899 non-null   float64
 6   key               1899 non-null   int64  
 7   liveness          1899 non-null   float64
 8   loudness          1899 non-null   float64
 9   audio_mode        1899 non-null   int64  
 10  speechiness       1899 non-null   float64
 11  tempo             1899 non-null   float64
 12  time_signature    1899 non-null   int64  
 13  audio_valence     1899 non-null   float64
dtypes: float64(9), int64(5)
memory usage: 207.8 KB
None


Шаг 2. Обработка выбросов

In [4]:
print(df.describe().T[['min', '50%', 'max']])

                           min            50%          max
song_popularity       0.000000      62.000000       99.000
song_duration_ms  63080.000000  227619.000000  1233666.000
acousticness          0.000003       0.141000        0.989
danceability          0.149000       0.583000        0.967
energy                0.020400       0.679000        0.995
instrumentalness      0.000000       0.000007        0.952
key                   0.000000       5.000000       11.000
liveness              0.021500       0.122000        0.986
loudness            -27.691000      -7.099000       -1.197
audio_mode            0.000000       1.000000        1.000
speechiness           0.022400       0.045300        0.891
tempo                62.330000     117.928000      213.226
time_signature        1.000000       4.000000        5.000
audio_valence         0.032900       0.554000        0.982


In [5]:
# 1. Отфильтруйте song_duration_ms (допустим: 60 секунд до 10 минут)
df = df[(df['song_duration_ms'] >= 60000) & (df['song_duration_ms'] <= 600000)]

# 2. Отфильтруйте loudness (оставляем разумный диапазон -20 до 0 дБ)
df = df[(df['loudness'] >= -20) & (df['loudness'] <= 0)]

# 3. Удалите строки с tempo < 30 BPM (слишком маловероятно)
df = df[(df['tempo'] >= 30)]

# 4. Удалите строки с time_signature == 0 (некорректный размер)
df = df[(df['time_signature'] != 0)]
print(df.shape)

(1886, 14)


Шаг 3. Масштабирование признаков

In [6]:
print(df.describe().T[['min', '50%', 'max']])

                           min            50%         max
song_popularity       0.000000      63.000000      99.000
song_duration_ms  63080.000000  227590.500000  547106.000
acousticness          0.000003       0.140000       0.989
danceability          0.149000       0.583000       0.967
energy                0.020400       0.680500       0.995
instrumentalness      0.000000       0.000007       0.952
key                   0.000000       5.000000      11.000
liveness              0.021500       0.121000       0.986
loudness            -19.846000      -7.083000      -1.197
audio_mode            0.000000       1.000000       1.000
speechiness           0.022400       0.045400       0.494
tempo                62.330000     117.881500     213.226
time_signature        1.000000       4.000000       5.000
audio_valence         0.037000       0.555000       0.982


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=['audio_valence']),
    df['audio_valence'],
    test_size=0.2,
    random_state=627
)

In [15]:
numeric_features = X_train.columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features)
    ]
)

Шаг 4. Построение базовой модели

In [16]:
pipe = Pipeline(
    steps=[
        ('preprocess', preprocessor),
        ('model', KNeighborsRegressor())
    ]
)

In [17]:
# Обучите модель
pipe_model = pipe.fit(X_train, y_train)

# Посчитайте MAE
mae_train = mean_absolute_error(y_train, pipe.predict(X_train))
mae_test = mean_absolute_error(y_test, pipe.predict(X_test))

# Вывод результата
print("MAE train:", round(mae_train, 3))
print("MAE test:", round(mae_test, 3))

MAE train: 0.131
MAE test: 0.157


Шаг 5. Подбор гиперпараметров kNN

In [18]:
param_grid = {
    'model__n_neighbors': [3, 5, 10, 15, 20, 25, 30],
    'model__weights': ['uniform', 'distance'],
    'model__metric': ['minkowski'],
    'model__p': [1, 2, 3, 4]
}

In [22]:
grid_search = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring='neg_mean_absolute_error',
)

grid_search.fit(X_train, y_train)

print("\nЛучшая комбинация параметров:")
print(grid_search.best_params_) # Выведите лучшую комбинацию
print("Лучшее значение MAE:", -grid_search.best_score_)


Лучшая комбинация параметров:
{'model__metric': 'minkowski', 'model__n_neighbors': 15, 'model__p': 1, 'model__weights': 'distance'}
Лучшее значение MAE: 0.14433975690057346


Шаг 6. Обучение лучшей модели

In [23]:
pipe.set_params(**grid_search.best_params_) 

,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [25]:
pipe.fit(X_train, y_train)

,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [26]:
# Посчитайте MAE
mae_train = mean_absolute_error(y_train, pipe.predict(X_train))
mae_test = mean_absolute_error(y_test, pipe.predict(X_test))

# Посчитайте MSE
mse_train = mean_squared_error(y_train, pipe.predict(X_train))
mse_test = mean_squared_error(y_test, pipe.predict(X_test))

# Посчитайте R2
r2_train =r2_score (y_train, pipe.predict(X_train))
r2_test = r2_score(y_test, pipe.predict(X_test))

In [27]:
results = pd.DataFrame({
    'data': ['train', 'test'],
    'MAE': [mae_train, mae_test],
    'MSE': [mse_train, mse_test],
    'R2': [r2_train, r2_test]
})
print(results)

    data           MAE           MSE       R2
0  train  4.417333e-19  4.904225e-35  1.00000
1   test  1.389137e-01  3.077122e-02  0.39284
